# Lab 11: VQE y Circuitos ParamétricosEl **Variational Quantum Eigensolver (VQE)** es el algoritmo híbrido cuántico-clásico prototípico de la era NISQ. Combina:- Un **circuito paramétrico** (ansatz) que prepara el estado de prueba $|\psi(\boldsymbol{\theta})\rangle$.- Una evaluación cuántica de la energía: $E(\boldsymbol{\theta}) = \langle\psi(\boldsymbol{\theta})|H|\psi(\boldsymbol{\theta})\rangle$.- Un **optimizador clásico** que ajusta $\boldsymbol{\theta}$ para minimizar $E$.El principio variacional garantiza $E(\boldsymbol{\theta}) \geq E_0$ para todo $\boldsymbol{\theta}$.

In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom scipy.optimize import minimizefrom qiskit import QuantumCircuitfrom qiskit.circuit import ParameterVectorfrom qiskit.circuit.library import TwoLocalfrom qiskit.quantum_info import SparsePauliOpfrom qiskit.primitives import StatevectorEstimatornp.random.seed(42)

## 1. Circuito Paramétrico ManualConstruimos a mano un ansatz de 2 capas: rotaciones $R_y(\theta_i)$ individuales y una capa de entrelazamiento CNOT.$$|\psi(\boldsymbol{\theta})\rangle = \text{CNOT}_{01} \cdot R_y(\theta_3)\otimes R_y(\theta_2) \cdot \text{CNOT}_{01} \cdot R_y(\theta_1)\otimes R_y(\theta_0)\, |00\rangle$$`ParameterVector` crea variables simbólicas que Qiskit sustituye en tiempo de ejecución.

In [ ]:
theta = ParameterVector('θ', 4)ansatz_manual = QuantumCircuit(2)ansatz_manual.ry(theta[0], 0)ansatz_manual.ry(theta[1], 1)ansatz_manual.cx(0, 1)ansatz_manual.ry(theta[2], 0)ansatz_manual.ry(theta[3], 1)ansatz_manual.cx(0, 1)print(f"Parámetros: {list(ansatz_manual.parameters)}")print(f"Profundidad: {ansatz_manual.depth()}")ansatz_manual.draw('mpl', style='clifford')

## 2. Hamiltoniano y Energía ExactaUsamos el modelo de Heisenberg-Ising de 2 espines:$$H = X_0 X_1 + Y_0 Y_1 + Z_0 Z_1$$Los valores propios exactos se obtienen diagonalizando la matriz $4\times 4$. El estado base tiene $E_0 = -3$.Verificamos el **principio variacional**: $\langle\psi(\boldsymbol{\theta})|H|\psi(\boldsymbol{\theta})\rangle \geq E_0$ para todo $\boldsymbol{\theta}$.

In [ ]:
H = SparsePauliOp.from_list([('XX', 1.0), ('YY', 1.0), ('ZZ', 1.0)])eigenvalues = np.linalg.eigvalsh(H.to_matrix())E0_exact = eigenvalues[0]print(f"Valores propios: {eigenvalues.round(4)}")print(f"Estado base exacto: E₀ = {E0_exact:.6f}")# Verificación del principio variacional con 1000 puntos aleatoriosestimator = StatevectorEstimator()energies = []for _ in range(100):    params = np.random.uniform(-np.pi, np.pi, 4)    e = float(estimator.run([(ansatz_manual, H, params)]).result()[0].data.evs)    energies.append(e)print(f"\nMín energía variacional obtenida: {min(energies):.6f}")print(f"Principio variacional respetado: {all(e >= E0_exact - 1e-9 for e in energies)}")

## 3. Bucle Híbrido VQEDefinimos la función de coste cuántica y usamos **COBYLA** (derivadas libres) para minimizar.El bucle:1. El optimizador propone $\boldsymbol{\theta}^{(k)}$.2. El simulador cuántico evalúa $E(\boldsymbol{\theta}^{(k)})$.3. El optimizador actualiza hasta convergencia.

In [ ]:
history = []def cost_vqe(params):    e = float(estimator.run([(ansatz_manual, H, params)]).result()[0].data.evs)    history.append(e)    return ex0 = np.random.uniform(-np.pi, np.pi, 4)result = minimize(cost_vqe, x0, method='COBYLA', options={'maxiter': 500, 'rhobeg': 0.5})print(f"Energía VQE:   {result.fun:.8f}")print(f"Energía exacta: {E0_exact:.8f}")print(f"Error:          {abs(result.fun - E0_exact):.2e}")print(f"Iteraciones:    {len(history)}")plt.figure(figsize=(8, 4))plt.plot(history, color='steelblue', lw=1.5, alpha=0.8)plt.axhline(E0_exact, color='red', ls='--', lw=2, label=f'$E_0$ exacta = {E0_exact:.4f}')plt.xlabel('Iteración', fontsize=12)plt.ylabel('Energía (a.u.)', fontsize=12)plt.title('Convergencia VQE — Heisenberg 2 espines')plt.legend()plt.tight_layout()plt.show()

## 4. Ansatz `TwoLocal` y ExpresividadEl ansatz `TwoLocal` de Qiskit parametriza de forma modular: bloques de rotación (p. ej. $R_y$) alternados con bloques de entrelazamiento (CNOT, CZ…). Al aumentar `reps` se incrementa la **expresividad** del circuito — su capacidad de aproximar estados arbitrarios en el espacio de Hilbert.Comparamos la precisión del ansatz manual (reps=1) con `TwoLocal` de más capas.

In [ ]:
results_twolocal = {}for reps in [1, 2, 3]:    ansatz_tl = TwoLocal(2, rotation_blocks='ry', entanglement_blocks='cx',                         entanglement='linear', reps=reps)    n = ansatz_tl.num_parameters    hist_tl = []    def cost_tl(params, a=ansatz_tl):        e = float(estimator.run([(a, H, params)]).result()[0].data.evs)        hist_tl.append(e)        return e    np.random.seed(reps)    x0_tl = np.random.uniform(-np.pi, np.pi, n)    res_tl = minimize(cost_tl, x0_tl, method='COBYLA', options={'maxiter': 1000})    results_twolocal[reps] = {'E': res_tl.fun, 'error': abs(res_tl.fun - E0_exact),                               'params': n, 'iters': len(hist_tl)}    print(f"reps={reps} | paráms={n:2d} | E={res_tl.fun:.6f} | error={abs(res_tl.fun-E0_exact):.2e}")print(f"\nE₀ exacta = {E0_exact:.6f}")

## 5. Gradientes: Parameter Shift RulePara circuitos con compuertas de la forma $G(\theta) = e^{-i\theta P/2}$ (con $P^2=I$), el gradiente es exacto:$$\frac{\partial E}{\partial\theta_k} = \frac{E(\theta_k + \pi/2) - E(\theta_k - \pi/2)}{2}$$Esta regla permite usar optimizadores basados en gradiente (p. ej. BFGS, Adam) sin diferenciación numérica.

In [ ]:
# Parameter shift para el parámetro θ[0] del ansatz manual# En el punto de los parámetros óptimos del VQEtheta_opt = result.x.copy()shift = np.pi / 2# Evaluaciones desplazadasdef E_eval(params):    return float(estimator.run([(ansatz_manual, H, params)]).result()[0].data.evs)grads_ps = []for k in range(4):    p_plus = theta_opt.copy(); p_plus[k] += shift    p_minus = theta_opt.copy(); p_minus[k] -= shift    g = (E_eval(p_plus) - E_eval(p_minus)) / 2    grads_ps.append(g)print("Gradientes (parameter shift) en θ_opt:")for k, g in enumerate(grads_ps):    print(f"  ∂E/∂θ_{k} = {g:.6f}")print(f"\n‖∇E‖ = {np.linalg.norm(grads_ps):.2e}  (≈ 0 en el mínimo)")